## Data Exploration and manipulation

In [1]:
import pandas as pd

df = pd.read_csv("/Users/ian/Programming/ML/FlexTrack_Challenge/flextrack-challenge-2025-starter-kit/data/flextrack-2025-training-data-v0.1.csv")

In [2]:
df[df["Demand_Response_Flag"] == 1].head()

,Site,Timestamp_Local,Dry_Bulb_Temperature_C,Global_Horizontal_Radiation_W/m2,Building_Power_kW,Demand_Response_Flag,Demand_Response_Capacity_kW
524,siteA,2019-01-06 11:00:00,18.75,964.0,17.19,1,7.68
525,siteA,2019-01-06 11:15:00,18.78,989.0,16.53,1,6.62
526,siteA,2019-01-06 11:30:00,18.82,1014.0,23.28,1,13.09
527,siteA,2019-01-06 11:45:00,18.86,1039.0,17.78,1,7.24
1012,siteA,2019-01-11 13:00:00,26.60,1095.0,15.62,1,1.26


In [3]:
df[df["Demand_Response_Flag"] == 0].head()

,Site,Timestamp_Local,Dry_Bulb_Temperature_C,Global_Horizontal_Radiation_W/m2,Building_Power_kW,Demand_Response_Flag,Demand_Response_Capacity_kW
0,siteA,2019-01-01 00:00:00,22.20,0.0,4.8,0,0.0
1,siteA,2019-01-01 00:15:00,22.27,0.0,4.8,0,0.0
2,siteA,2019-01-01 00:30:00,22.35,0.0,4.8,0,0.0
3,siteA,2019-01-01 00:45:00,22.42,0.0,4.8,0,0.0
4,siteA,2019-01-01 01:00:00,22.50,0.0,4.8,0,0.0


In [4]:
df[df["Demand_Response_Flag"] == -1].head()

,Site,Timestamp_Local,Dry_Bulb_Temperature_C,Global_Horizontal_Radiation_W/m2,Building_Power_kW,Demand_Response_Flag,Demand_Response_Capacity_kW
528,siteA,2019-01-06 12:00:00,18.89,1064.0,13.09,-1,0.65
529,siteA,2019-01-06 12:15:00,18.95,1072.0,10.18,-1,-0.48
530,siteA,2019-01-06 12:30:00,19.00,1081.0,10.25,-1,1.01
531,siteA,2019-01-06 12:45:00,19.05,1090.0,8.05,-1,-1.37
532,siteA,2019-01-06 13:00:00,19.10,1098.0,8.49,-1,-1.49


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 105120 entries, 0 to 105119
Data columns (total 7 columns):
 #   Column                            Non-Null Count   Dtype  
---  ------                            --------------   -----  
 0   Site                              105120 non-null  object 
 1   Timestamp_Local                   105120 non-null  object 
 2   Dry_Bulb_Temperature_C            105120 non-null  float64
 3   Global_Horizontal_Radiation_W/m2  105120 non-null  float64
 4   Building_Power_kW                 105120 non-null  float64
 5   Demand_Response_Flag              105120 non-null  int64  
 6   Demand_Response_Capacity_kW       105120 non-null  float64
dtypes: float64(4), int64(1), object(2)
memory usage: 5.6+ MB


In [6]:
df["Site"].unique()

array(['siteA', 'siteB', 'siteC'], dtype=object)

### Enhancing the Timestamp_local Feature

To make the Timestamp_local usable for the model, we transform it using periodic functions with different cycle lengths. This helps the model capture time-based patterns, and then we can identify which periods are most effective for improving prediction accuracy.

Can experiment with different cycle lengths too

In [7]:
import numpy as np
from random import random

def encode_time(df):
    df["Timestamp_Local"] = pd.to_datetime(df["Timestamp_Local"])
    df["hour_of_day"] = df["Timestamp_Local"].dt.hour + df["Timestamp_Local"].dt.minute / 60
    df["day_of_week"] = df["Timestamp_Local"].dt.weekday

    # 1 day periodic
    df["hour_sin"] = np.sin(df["hour_of_day"] * (2 * np.pi / 24))
    df["hour_cos"] = np.cos(df["hour_of_day"] * (2 * np.pi / 24))

    # 1 week periodic
    df["weekday_sin"] = np.sin(df["day_of_week"] * (2 * np.pi / 7))
    df["weekday_cos"] = np.cos(df["day_of_week"] * (2 * np.pi / 7))

encode_time(df)

## Experimenting

In [8]:
from sklearn.metrics import confusion_matrix

def GMS(y_test, y_pred):
    cm = confusion_matrix(y_test, y_pred)
    recalls = []
    
    for i in range(len(cm)):
        tp = cm[i, i]
        fn = np.sum(cm[i, :]) - tp
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        recalls.append(recall)

    gmean = np.prod(recalls) ** (1.0 / len(recalls)) if recalls else 0
    return gmean

In [9]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# site = "siteB"
# df = df[df["Site"] == site]

features = [
    "Dry_Bulb_Temperature_C", "Global_Horizontal_Radiation_W/m2", "Building_Power_kW",
    "hour_sin", "hour_cos", "weekday_sin", "weekday_cos", 
]
labels = "Demand_Response_Flag"

X = df[features].to_numpy()
y = df[labels].to_numpy()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

### Logistic Regression

In [10]:
# from sklearn.linear_model import LogisticRegression
# from sklearn.feature_selection import SelectFromModel
# from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

# clf = LogisticRegression(
#     penalty="l1", 
#     solver="saga", 
#     multi_class="multinomial", 
#     C=0.01, 
#     max_iter=5000,
#     class_weight="balanced"
# )

# clf.fit(X_train, y_train)

# selector = SelectFromModel(clf, prefit=True)
# print(selector.get_support())
# print(clf.coef_)

# y_pred = clf.predict(X_test) 
# print("Geometric Mean Score:", GMS(y_test, y_pred))
# print("\nClassification Report:\n", classification_report(y_test, y_pred))
# print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))


### Ensembles

In [11]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.feature_selection import SelectFromModel
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=10)

RFC = RandomForestClassifier(
    random_state=42,
    class_weight="balanced" 
)

RFC.fit(X_train, y_train)
y_pred = RFC.predict(X_test)

print("Geometric Mean Score:", GMS(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))


Geometric Mean Score: 0.3978060314651395

Classification Report:
               precision    recall  f1-score   support

          -1       0.85      0.26      0.39       707
           0       0.98      1.00      0.99     30565
           1       0.89      0.25      0.39       264

    accuracy                           0.98     31536
   macro avg       0.91      0.50      0.59     31536
weighted avg       0.97      0.98      0.97     31536



In [12]:
from imbens.ensemble import BalanceCascadeClassifier
BCC = BalanceCascadeClassifier(random_state=41)
BCC.fit(X_train, y_train)

y_pred = BCC.predict(X_test)
print("Geometric Mean Score:", GMS(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Geometric Mean Score: 0.8899017239301418

Classification Report:
               precision    recall  f1-score   support

          -1       0.13      0.95      0.22       707
           0       1.00      0.80      0.89     30565
           1       0.15      0.92      0.26       264

    accuracy                           0.81     31536
   macro avg       0.42      0.89      0.46     31536
weighted avg       0.97      0.81      0.87     31536



In [13]:
# Train an SPE classifier
from imbens.ensemble import BalancedRandomForestClassifier
BRFC = BalancedRandomForestClassifier(random_state=41)
BRFC.fit(X_train, y_train)

# Predict with an SPE classifier
y_pred = BRFC.predict(X_test)
print("Geometric Mean Score:", GMS(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Geometric Mean Score: 0.8672380533654095

Classification Report:
               precision    recall  f1-score   support

          -1       0.11      0.94      0.20       707
           0       1.00      0.74      0.85     30565
           1       0.08      0.94      0.14       264

    accuracy                           0.74     31536
   macro avg       0.40      0.87      0.40     31536
weighted avg       0.97      0.74      0.83     31536



In [14]:
# Train an SPE classifier
from imbens.ensemble import SelfPacedEnsembleClassifier
SPE = SelfPacedEnsembleClassifier(random_state=41)
SPE.fit(X_train, y_train)

# Predict with an SPE classifier
y_pred = SPE.predict(X_test)
print("Geometric Mean Score:", GMS(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Geometric Mean Score: 0.9124640161892974

Classification Report:
               precision    recall  f1-score   support

          -1       0.21      0.94      0.35       707
           0       1.00      0.88      0.94     30565
           1       0.17      0.91      0.29       264

    accuracy                           0.88     31536
   macro avg       0.46      0.91      0.52     31536
weighted avg       0.97      0.88      0.92     31536



In [15]:
from imbens.ensemble import SMOTEBaggingClassifier   
SBC = SMOTEBaggingClassifier(random_state=41)
SBC.fit(X_train, y_train)

y_pred = SBC.predict(X_test)
print("Geometric Mean Score:", GMS(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Geometric Mean Score: 0.5929257628298299

Classification Report:
               precision    recall  f1-score   support

          -1       0.78      0.51      0.62       707
           0       0.98      1.00      0.99     30565
           1       0.70      0.41      0.52       264

    accuracy                           0.98     31536
   macro avg       0.82      0.64      0.71     31536
weighted avg       0.98      0.98      0.98     31536



In [16]:
from imbens.ensemble import OverBaggingClassifier   
OBC = OverBaggingClassifier(random_state=42)
OBC.fit(X_train, y_train)

y_pred = OBC.predict(X_test)
print("Geometric Mean Score:", GMS(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion matrix:\n", confusion_matrix(y_test, y_pred))

Geometric Mean Score: 0.5404081692870415

Classification Report:
               precision    recall  f1-score   support

          -1       0.83      0.44      0.57       707
           0       0.98      1.00      0.99     30565
           1       0.77      0.36      0.49       264

    accuracy                           0.98     31536
   macro avg       0.86      0.60      0.68     31536
weighted avg       0.98      0.98      0.98     31536


Confusion matrix:
 [[  311   395     1]
 [   64 30474    27]
 [    1   168    95]]


### K-NN

## Testing

In [17]:
clf = BRFC # options: RFC, SPE, SBC, OBC, BCC, BRFC

test_df = pd.read_csv("/Users/ian/Programming/ML/FlexTrack_Challenge/flextrack-challenge-2025-starter-kit/data/flextrack-2025-public-test-data-v0.1.csv")
orig_test_df = test_df.copy()
encode_time(test_df)
# test_df = test_df[test_df["Site"] == site]
X_test_real = test_df[features].to_numpy()
X_test_real = scaler.transform(X_test_real)

y_pred_real = clf.predict(X_test_real)
orig_test_df["Demand_Response_Flag"] = y_pred_real
orig_test_df["Demand_Response_Flag"].unique()

array([0])

In [18]:
orig_test_df["Demand_Response_Flag"].unique()


array([0])

In [19]:
orig_test_df.to_csv("/Users/ian/Programming/ML/FlexTrack_Challenge/flextrack-challenge-2025-starter-kit/data/flextrack-2025-attempted-prediction-data-v0.1.csv")